# 🔍 Visualización de Anomalías en Runs Nuevas (Evaluación)

Este módulo interactivo permite analizar los resultados del modelo aplicado a datos no vistos (evaluación), explorando:

- Señal `CCL` suavizada.
- Score de anomalía (Isolation Forest) por tramos.
- Tramos más anómalos resaltados.
- Exportación profesional de resultados.


### 💾 Celda 2 – Carga del dataset de evaluación

In [1]:
import pandas as pd

# Cargar resultados
df_eval = pd.read_csv(r"C:\Developer\fundamentos\data\ccl_eval_scores.csv")

# Asegurar orden correcto
df_eval = df_eval.sort_values(by=["pozo", "etapa", "DEPT"]).reset_index(drop=True)

# Verificación
print(f"Total de filas: {len(df_eval)}")
df_eval.head()


Total de filas: 28617


,DEPT,CCL,TENS,archivo_origen,pozo,sentido,etapa,CCL_norm,dCCL,abs_dCCL,...,CCL_norm_max_y,CCL_norm_min_y,abs_dCCL_mean_y,abs_dCCL_std_y,abs_dCCL_max_y,TENS_mean_y,TENS_std_y,TENS_max_y,score_iso,anomaly_iso
0,2800.05,0.00165,736.99993,BPE-2343_E38_Down__24Nov24_013757.las,BPE-2343,Down,E38,0.330132,NaN,NaN,...,10.0,-9.92397,2.864947,2.864363,19.485794,1207.128372,261.106938,1716.29916,-0.078496,1
1,2800.10,-0.00068,736.99993,BPE-2343_E38_Down__24Nov24_013757.las,BPE-2343,Down,E38,-0.136054,-0.466186,0.466186,...,10.0,-9.92397,2.864947,2.864363,19.485794,1207.128372,261.106938,1716.29916,-0.085431,1
2,2800.15,0.00474,736.99993,BPE-2343_E38_Down__24Nov24_013757.las,BPE-2343,Down,E38,0.948379,1.084434,1.084434,...,10.0,-9.92397,2.864947,2.864363,19.485794,1207.128372,261.106938,1716.29916,-0.077626,1
3,2800.20,0.00902,736.99993,BPE-2343_E38_Down__24Nov24_013757.las,BPE-2343,Down,E38,1.804722,0.856343,0.856343,...,10.0,-9.92397,2.864947,2.864363,19.485794,1207.128372,261.106938,1716.29916,-0.112709,1
4,2800.25,0.00835,736.99993,BPE-2343_E38_Down__24Nov24_013757.las,BPE-2343,Down,E38,1.670668,-0.134054,0.134054,...,10.0,-9.92397,2.864947,2.864363,19.485794,1207.128372,261.106938,1716.29916,-0.103661,1


### 🧰 Celda 3 – Herramientas interactivas

In [2]:
import plotly.graph_objs as go
import plotly.colors as colors
import ipywidgets as widgets
from IPython.display import display

# Dropdowns dinámicos
pozo_dropdown = widgets.Dropdown(options=sorted(df_eval["pozo"].unique()), description="Pozo:")
etapa_dropdown = widgets.Dropdown(description="Etapa:")

def update_etapas(pozo_sel):
    etapas = df_eval[df_eval["pozo"] == pozo_sel]["etapa"].unique()
    etapa_dropdown.options = sorted(etapas)

pozo_dropdown.observe(lambda change: update_etapas(change["new"]), names="value")
update_etapas(pozo_dropdown.value)

step_dropdown = widgets.Dropdown(options=[2.5, 5, 10, 15, 25, 30], value=30, description="Paso (m):")

display(pozo_dropdown, etapa_dropdown, step_dropdown)


Dropdown(description='Pozo:', options=('BPE-2343',), value='BPE-2343')

Dropdown(description='Etapa:', options=('E38',), value=None)

Dropdown(description='Paso (m):', index=5, options=(2.5, 5, 10, 15, 25, 30), value=30)

### 📊 Celda 4 – Función de visualización completa

In [3]:
def plot_eval_riesgo(df, pozo, etapa, step):
    df_et = df[(df["pozo"] == pozo) & (df["etapa"] == etapa)].sort_values("DEPT").copy()
    df_et["CCL_smooth"] = df_et["CCL"].rolling(window=10, min_periods=1).mean()
    bin_col = f"DEPT_bin_{step}"
    df_et[bin_col] = (df_et["DEPT"] // step) * step

    # Score promedio por tramo
    grouped = df_et.groupby(bin_col)["score_iso"].mean().reset_index().rename(columns={"score_iso": f"score_mean_{step}"})
    score_col = f"score_mean_{step}"
    top5 = grouped.sort_values(score_col, ascending=False).head(10).reset_index(drop=True)

    # Colores según severidad
    max_s = top5[score_col].max()
    min_s = top5[score_col].min()
    scale = colors.sequential.OrRd

    def s2color(score):
        idx = int(((score - min_s) / (max_s - min_s + 1e-5)) * (len(scale) - 1))
        return scale[idx]

    # Gráfico
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_et["CCL_smooth"],
        y=df_et["DEPT"],
        mode='lines',
        name='CCL suavizado',
        line=dict(color='blue')
    ))

    fig.add_trace(go.Scatter(
        x=grouped[score_col],
        y=grouped[bin_col],
        mode='lines+markers',
        name=f'Score medio cada {step}m',
        line=dict(color='black', width=2),
        marker=dict(size=6)
    ))

    for _, row in top5.iterrows():
        y0 = row[bin_col]
        y1 = y0 + step
        fig.add_shape(
            type="rect", x0=0, x1=1, xref="paper",
            y0=y0, y1=y1, yref="y",
            fillcolor=s2color(row[score_col]), opacity=0.3,
            line_width=0, layer="below"
        )

    fig.update_layout(
        title=f"Pozo: {pozo} | Etapa: {etapa} | Paso: {step} m",
        xaxis_title="Valor",
        yaxis_title="Profundidad (DEPT)",
        yaxis_autorange="reversed",
        height=700,
        margin=dict(l=20, r=20, t=50, b=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        font=dict(family="Arial", size=14)
    )

    fig.show()

    print(f"📋 Top 5 tramos más anómalos ({step} m):")
    display(top5)

    return fig, top5


### 💾 Celda 5 – Ejecutar visualización y exportar informe

In [4]:
fig, top5 = plot_eval_riesgo(df_eval, pozo_dropdown.value, etapa_dropdown.value, step_dropdown.value)

# Exportar si querés
#nombre_archivo = f"reporte_eval_{pozo_dropdown.value}_etapa{etapa_dropdown.value}_{step_dropdown.value}m.html"
#fig.write_html(nombre_archivo)
#print(f"✅ Gráfico exportado a: {nombre_archivo}")

# Guardar tabla top5
#top5.to_csv(nombre_archivo.replace(".html", "_top5.csv"), index=False)


📋 Top 5 tramos más anómalos (25 m):


,DEPT_bin_25,score_mean_25
0,4275.0,-0.024071
1,4175.0,-0.026016
2,4225.0,-0.032033
3,4150.0,-0.039857
4,4025.0,-0.040751
5,4200.0,-0.042507
6,4000.0,-0.043170
7,3950.0,-0.046159
8,3975.0,-0.046748
9,4075.0,-0.048677
